# Behavioral Prediction LSTM

This notebook implements the LSTM that predicts cortical NE dynamics
from behavioral variables (pupil dilation and locomotion) alone, as
described in:
> **Inferring norepinephrine dynamics from partial observations reveals the temporal structure of elevations during arousal — Erin Neyhart, Brandon R. Munn, Na Zhou, Peilin Yang, Jiesi Feng, Yulong Li, James M. Shine, Jacob Reimer**

## Overview

When no direct recording of NE has been conducted, behavioral variables can be used to recover key features of NE dynamics. This notebook trains a single-model LSTM that
takes as input:
- Pupil radius
- d/dt pupil radius
- Locomotion speed
- d/dt locomotion speed

and predicts the NE fluorescence signal.

## Notebook structure

1. Imports and configuration
2. Model architecture
3. Predicting raw (uncorrected) NE from behavior
4. Evaluation and visualization

## Data

Input CSVs have the same format as described in `01_hemodynamic_correction_lstm.ipynb`, with the exception that the filenames are in the format `{animal_id}_{session}_{scan_idx}.csv`.
See `data/README.md` for the dataset DOI and column descriptions.


## 1. Imports and configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import random
import os
import glob
from scipy.stats import pearsonr

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from utils import (
    find_nearest,
    interpolate_to_new_timestamps,
    interpolate_nans,
    preprocess_dFF,
    lowpass_butter,
    lowpass_hamming,
    median_filter,
    safe_zscore,
    correct_regression_no_runs,
)

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [2]:
# ---------------------------------------------------------------------------
# Configuration — edit these paths before running
# ---------------------------------------------------------------------------

# Directory containing per-recording CSVs (same format as notebook 01)
# DATA_DIR = "data/NE-bx_prediction/"
DATA_DIR = "/mnt/lab/users/erin/data_to_export/NE-bx_prediction/"  ###########change this

# Output directory for this model's weights and train/test data
OUTPUT_DIR = "output/"

# Preprocessing parameters
LOWPASS_CUTOFF = 1.0      # Hz
WINDOW_SEC     = 2 * 60   # s — 2-minute chunks
MAX_RATIO      = 5        # dF/F quality guard

# Training parameters
N_EPOCHS   = 50
BATCH_SIZE = 32
LR         = 1e-3
HIDDEN_SIZE = 256
FC_SIZE     = 256
DROPOUT     = 0.1
SEED        = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Model architecture

Same architecture as the hemodynamic correction LSTM (notebook 01).
Input size varies by section: 3 for raw NE (radius, d/dt radius, velocity),
4 for corrected NE (adds d/dt velocity).

In [3]:
class LSTMRegressor(nn.Module):
    """LSTM-based sequence-to-sequence regressor.

    Parameters
    ----------
    input_size : int
        Number of input features per time step.
    hidden_size : int
        LSTM hidden state size.
    fc_size : int
        Size of the intermediate fully connected layer.
    dropout_p : float
        Dropout probability applied between FC layers.
    output_size : int
        Output features per time step (1 for scalar regression).
    """

    def __init__(self, input_size, hidden_size=256, fc_size=256,
                 dropout_p=0.1, output_size=1):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc1     = nn.Linear(hidden_size, fc_size)
        self.dropout = nn.Dropout(dropout_p)
        self.fc2     = nn.Linear(fc_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)           # (batch, seq_len, hidden)
        out = torch.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)
        return out.squeeze(-1)          # (batch, seq_len)


def pearson_loss(y_pred, y_true):
    """Differentiable Pearson correlation loss (minimized when r → 1)."""
    vx = y_pred - y_pred.mean(dim=1, keepdim=True)
    vy = y_true - y_true.mean(dim=1, keepdim=True)
    cost = -(vx * vy).sum(dim=1) / (
        torch.sqrt((vx ** 2).sum(dim=1)) *
        torch.sqrt((vy ** 2).sum(dim=1)) + 1e-8
    )
    return cost.mean()

---
## 3. Main figure: predicting raw (uncorrected) NE from behavior

Target: z-scored raw NE fluorescence (dF/F, not hemodynamically corrected).  
Predictors: pupil radius, d/dt pupil radius, locomotion speed (3 features).  
Loss: Pearson correlation loss.

The train/test split is held out at the **recording level** to prevent
data leakage across 2-minute chunks from the same session.

In [27]:
# ---------------------------------------------------------------------------
# Load and preprocess data for NE prediction from behavioral variables
# ---------------------------------------------------------------------------

EXCLUDED_RECORDINGS = {
    # add any recordings to exclude here
}

# sensor one-hot encoding — 3 labels to match saved model weights (input size 7)
sensor_map = {'rNE0.5': 0, 'rNE1.0': 1, 'GrabNE2h': 2}

# build file list from two folders, inferring sensor type from folder
RNE_DATA_DIR     = os.path.join(DATA_DIR, "rNE0.5/")
GRABNE_DATA_DIR = os.path.join(DATA_DIR, "GrabNE2h/")

all_files = (
    [(f, 'rNE0.5')   for f in glob.glob(os.path.join(RNE_DATA_DIR,     "*.csv"))] +
    [(f, 'GrabNE2h') for f in glob.glob(os.path.join(GRABNE_DATA_DIR, "*.csv"))]
)

alldat = []

for filepath, sensor_label in all_files:
    filename = os.path.splitext(os.path.basename(filepath))[0]

    if filename in EXCLUDED_RECORDINGS:
        continue

    sensor_onehot = np.zeros(len(sensor_map))
    sensor_onehot[sensor_map[sensor_label]] = 1

    # --- load CSV ---
    try:
        df = pd.read_csv(filepath)
    except:
        print('no main file ' + str(filename))
        continue

    fl_times     = df['fluorescence_timestamps'].values
    fl_sensor    = df['fluorescence'].values
    velocity     = df['velocity_in_cm_per_sec'].values
    vel_times    = df['velocity_timestamps'].values
    radius       = df['radius_in_pixels'].values
    radius_times = df['radius_timestamps'].values

    # pre-process to match database preprocessing
    velocity    = np.abs(velocity)
    velocity    = median_filter(velocity, 1.0 / np.nanmedian(np.diff(vel_times)), 0.5)

    # cut/align
    scan_start_secs = fl_times[np.isfinite(fl_times)][0] + 20
    scan_stop_secs  = fl_times[np.isfinite(fl_times)][-1]

    start     = find_nearest(fl_times, scan_start_secs)
    stop      = find_nearest(fl_times, scan_stop_secs)
    fl_times  = fl_times[start:stop]
    fl_sensor = fl_sensor[start:stop]

    start     = find_nearest(vel_times, scan_start_secs)
    stop      = find_nearest(vel_times, scan_stop_secs)
    vel_times = vel_times[start:stop]
    velocity  = velocity[start:stop]

    start        = find_nearest(radius_times, scan_start_secs)
    stop         = find_nearest(radius_times, scan_stop_secs)
    radius_times = radius_times[start:stop]
    radius       = radius[start:stop]

    if np.any(np.isnan(radius_times)): radius_times = interpolate_nans(radius_times.copy())
    if np.any(np.isnan(radius)):       radius        = interpolate_nans(radius.copy())
    pupilfps  = round(1.0 / np.nanmedian(np.diff(radius_times)))
    new_times = radius_times

    fps       = 1.0 / np.nanmedian(np.diff(fl_times))

    # preprocess NE fluorescence
    try:
        fl_NE = preprocess_dFF(fl_sensor, fps, max_ratio=5, filt=True, interp=True,
                               lowpass_cutoff=LOWPASS_CUTOFF, old_times=fl_times,
                               new_times=new_times)
    except:
        print('negative dF/F baseline; skipping ' + str(filename))
        continue

    # preprocess radius — NaN interpolation already done above, now filter
    # note: px_to_mm conversion omitted — radius is z-scored so units are irrelevant
    radius    = lowpass_hamming(radius, pupilfps, lowpass_freq=1.0)
    radius_dv = np.gradient(radius)

    # preprocess velocity — abs, median filter, lowpass filter, interpolate
    velocity    = lowpass_butter(velocity, 100, 1) 
    velocity    = interpolate_to_new_timestamps(vel_times, velocity, new_times)
    velocity_dv = np.gradient(velocity)

    if np.any(np.isnan(velocity)):
        print(f'NaN in velocity for {filename}')
        continue

    if np.any(np.isnan(velocity_dv)):
        print(f'NaN in velocity_dv for {filename}')
        continue

    # z-score full traces before chunking
    fl_NE_z       = (fl_NE       - np.nanmean(fl_NE))       / (np.nanstd(fl_NE)       + 1e-8)
    radius_z      = (radius      - np.nanmean(radius))      / (np.nanstd(radius)      + 1e-8)
    radius_dv_z   = (radius_dv   - np.nanmean(radius_dv))   / (np.nanstd(radius_dv)   + 1e-8)
    velocity_z    = (velocity    - np.nanmean(velocity))    / (np.nanstd(velocity)    + 1e-8)
    velocity_dv_z = (velocity_dv - np.nanmean(velocity_dv)) / (np.nanstd(velocity_dv) + 1e-8)

    # cut into 2-minute chunks
    ndt   = int(WINDOW_SEC * pupilfps)
    n2min = int(len(new_times) // ndt)

    for NN in range(n2min):
        start = NN * ndt
        end   = (NN + 1) * ndt
        tclip = slice(start, end)

        NE_chunk  = fl_NE_z[tclip]

        beh_chunk = np.vstack([
            radius_z[tclip],
            radius_dv_z[tclip],
            velocity_z[tclip],
            velocity_dv_z[tclip],
        ])  # shape: (4, ndt)

        sensor_tiled = np.tile(sensor_onehot, (beh_chunk.shape[1], 1)).T  # (3, ndt)
        predictor    = np.vstack([beh_chunk, sensor_tiled])                # (7, ndt)

        alldat.append([NE_chunk, predictor, filename])

unique_recs = list(set([alldat[i][2] for i in range(len(alldat))]))
print(f"Total unique scans: {len(unique_recs)}")
print(f"Total 2-min segments: {len(alldat)}")

negative dF/F baseline; skipping 27195_3_1
negative dF/F baseline; skipping 22708_4_4
negative dF/F baseline; skipping 22780_10_2
negative dF/F baseline; skipping 22780_13_1
negative dF/F baseline; skipping 22780_17_1
negative dF/F baseline; skipping 22780_19_1
negative dF/F baseline; skipping 22780_19_2
negative dF/F baseline; skipping 22780_19_3
negative dF/F baseline; skipping 23332_13_1
negative dF/F baseline; skipping 23332_2_1
negative dF/F baseline; skipping 23332_7_1
negative dF/F baseline; skipping 23337_5_1
negative dF/F baseline; skipping 25791_3_1
Total unique scans: 129
Total 2-min segments: 1431


In [20]:
# ---------------------------------------------------------------------------
# Train/test split by recording (random, ~10% test)
# ---------------------------------------------------------------------------
total_num   = len(alldat)
unique_recs = list(set([alldat[i][2] for i in range(total_num)]))
test_recs   = set(random.sample(unique_recs, round(0.1 * len(unique_recs))))
leaveOutRecs = [i for i in range(total_num)
                    if alldat[i][2] in test_recs]

train_x, train_y, test_x, test_y = [], [], [], []
for i in range(total_num):
    NE  = alldat[i][0]
    beh = alldat[i][1].T   # (ndt, 7)
    if i in leaveOutRecs:
        test_x.append(beh)
        test_y.append(NE)
    else:
        train_x.append(beh)
        train_y.append(NE)

X_train_raw = np.stack(train_x, axis=0)   # (N_train, ndt, 7)
y_train_raw = np.stack(train_y, axis=0)
X_test_raw  = np.stack(test_x,  axis=0)
y_test_raw  = np.stack(test_y,  axis=0)

X_train_t = torch.tensor(X_train_raw, dtype=torch.float32)
y_train_t = torch.tensor(y_train_raw, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_raw,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test_raw,  dtype=torch.float32)

print(f"Train chunks: {X_train_raw.shape[0]}  |  Test chunks: {X_test_raw.shape[0]}")

Train chunks: 1308  |  Test chunks: 123


In [ ]:
# ---------------------------------------------------------------------------
# Train: raw NE prediction model
# ---------------------------------------------------------------------------
loader_raw = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE, shuffle=True
)

model_raw = LSTMRegressor(
    input_size=X_train_raw.shape[-1],   # 3
    hidden_size=HIDDEN_SIZE, fc_size=FC_SIZE, dropout_p=DROPOUT
)
optimizer_raw = optim.Adam(model_raw.parameters(), lr=LR)

print("Training raw NE prediction model...")
for epoch in range(N_EPOCHS):
    model_raw.train()
    total_loss = 0
    for X_batch, y_batch in loader_raw:
        optimizer_raw.zero_grad()
        loss = pearson_loss(model_raw(X_batch), y_batch)
        loss.backward()
        optimizer_raw.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/{N_EPOCHS}  loss={total_loss/len(loader_raw):.4f}")

# Save weights and data
torch.save(model_raw.state_dict(),
           os.path.join(OUTPUT_DIR, "NE_bx_model_raw_weights.pth"))
with open(os.path.join(OUTPUT_DIR, "NE_bx_model_raw_train_test_data.pkl"), "wb") as f:
    pickle.dump({
        "X_train": X_train_raw, "y_train": y_train_raw,
        "X_test":  X_test_raw,  "y_test":  y_test_raw,
        "leaveOutRecs": leaveOutRecs,
    }, f)
print("Saved.")

#### ---
## 4. Evaluation and visualization

In [ ]:
def compute_metrics(model, X, y):
    """Per-chunk Pearson r and RMSE."""
    model.eval()
    corrs, rmses = [], []
    with torch.no_grad():
        for i in range(X.shape[0]):
            y_true = y[i].numpy()
            y_pred = model(X[i].unsqueeze(0)).squeeze(0).numpy()
            if np.all(np.isnan(y_true)) or np.all(np.isnan(y_pred)):
                continue
            r, _  = pearsonr(y_true, y_pred)
            rmse  = np.sqrt(np.mean((y_true - y_pred) ** 2))
            corrs.append(r)
            rmses.append(rmse)
    return np.array(corrs), np.array(rmses)


# Raw NE model
raw_train_corr, raw_train_rmse = compute_metrics(model_raw, X_train_t, y_train_t)
raw_test_corr,  raw_test_rmse  = compute_metrics(model_raw, X_test_t,  y_test_t)


print("Raw NE model (main figure):")
print(f"  Train  r={np.median(raw_train_corr):.3f}  RMSE={np.median(raw_train_rmse):.4f}")
print(f"  Test   r={np.median(raw_test_corr):.3f}  RMSE={np.median(raw_test_rmse):.4f}")
print()

In [ ]:
# Example trace plots 
n_examples = min(5, X_test_t.shape[0])
fig, axes  = plt.subplots(n_examples, 1, figsize=(12, 3 * n_examples), sharex=True)

model_raw.eval()
with torch.no_grad():
    for i, ax in enumerate(axes):
        y_true = y_test_t[i].numpy()
        y_pred = model_raw(X_test_t[i].unsqueeze(0)).squeeze(0).numpy()
        t = np.arange(len(y_true))
        ax.plot(t, y_true, color="steelblue",  label="True NE",         linewidth=1.0)
        ax.plot(t, y_pred, color="darkorange",  label="Predicted",       linewidth=1.0,
                linestyle="--")
        ax.set_ylabel("NE (z-score)")
        if i == 0:
            ax.legend()

axes[-1].set_xlabel("Time (frames)")
fig.suptitle("Behavioral prediction model — raw NE (test set)")
plt.tight_layout()
plt.show()

## Reload saved models (for inference without retraining)

In [ ]:
# Raw NE model (input_size=3)
model_raw_loaded = LSTMRegressor(
    input_size=3, hidden_size=HIDDEN_SIZE, fc_size=FC_SIZE, dropout_p=DROPOUT
)
model_raw_loaded.load_state_dict(
    torch.load(os.path.join(OUTPUT_DIR, "NE_bx_model_raw_weights.pth"),
               map_location="cpu")
)
model_raw_loaded.eval()

print("Models loaded.")